# TB-MTNet: Multi-Task Hybrid CNN-Transformer
## TB Detection + Timika Severity Regression on Chest X-Rays

**Architecture:** Inception-v3 → ECA → 2048→512→96 bridge → 4-layer Transformer → dual heads  
**Targets:** AUROC ≥ 0.92 | MAE ≤ 14 Timika points  
**Compute:** Kaggle T4 ×2 | fp16 | batch 32 / 512²  

### Required Kaggle datasets (Add Data → search each slug):
```
raddar/tuberculosis-chest-xrays-shenzhen
yoctoman/shcxr-lung-mask
raddar/tuberculosis-chest-xrays-montgomery
vbookshelf/tbx11k-simplified
(optional) tawsifurrahman/tuberculosis-tb-chest-xray-dataset
```

### Training schedule:
| Stage | Epochs | What trains |
|-------|--------|-------------|
| 1 | 10 | Trunk + Transformer + cls head (reg head frozen) |
| 2 | 20 | Full multi-task (Kendall–Gal–Cipolla uncertainty weighting) |
| 3 | 10 | Fine-tune + SWA last 5 epochs |


## Cell 1 — Imports · Seed · Config

In [ ]:
# !pip install -q timm albumentations segmentation-models-pytorch grad-cam

import os, json, math, random, warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, List, Tuple, Dict

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torch.optim.swa_utils import AveragedModel, SWALR

import timm
from timm.scheduler import CosineLRScheduler
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, confusion_matrix
from scipy.stats import pearsonr, spearmanr

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED = 42
def seed_everything(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False  # required for full reproducibility
seed_everything()

@dataclass
class Config:
    SEED:              int  = 42
    SHENZHEN_DIR:      Path = Path("/kaggle/input/tuberculosis-chest-xrays-shenzhen")
    SHENZHEN_MASK_DIR: Path = Path("/kaggle/input/shcxr-lung-mask")
    MONTGOMERY_DIR:    Path = Path("/kaggle/input/tuberculosis-chest-xrays-montgomery")
    TBX11K_DIR:        Path = Path("/kaggle/input/tbx11k-simplified")
    RAHMAN_DIR:        Path = Path("/kaggle/input/tuberculosis-tb-chest-xray-dataset")
    SHEN_ANNOT_JSON:   Path = Path("/kaggle/input/datasets/manishchoudhary9/shenzen-polygon-annotations/Annotations_AllinOne_json.json")
    BASE:       Path = Path("/kaggle/working")
    CKPT_DIR:   Path = Path("/kaggle/working/checkpoints")
    LABELS_CSV: Path = Path("/kaggle/working/labels.csv")
    LUNG_CKPT:  Path = Path("/kaggle/working/checkpoints/lung_unet.pt")
    IMAGE_SIZE: int   = 512
    SEG_SIZE:   int   = 256
    D_MODEL:    int   = 96
    N_HEADS:    int   = 4
    N_LAYERS:   int   = 4
    FFN_DIM:    int   = 384
    DROPOUT:    float = 0.1
    # ── Multi-GPU (T4 x2) settings ──────────────────────────────────
    NUM_GPUS:     int   = torch.cuda.device_count()          # 2 on T4 x2
    # 48 samples per GPU × 2 GPUs = 96 effective batch size
    # T4 has 16 GB VRAM; Inception-v3 + Transformer at 512^2 fits comfortably
    BATCH_SIZE:   int   = 96
    # 3 workers per GPU → 6 total; extra workers saturate CLAHE+ElasticTransform
    NUM_WORKERS:  int   = 6
    N_FOLDS:      int   = 5                                  # Full 5-fold CV
    POS_WEIGHT:   float = 5.95
    HUBER_BETA:   float = 0.1
    SEVERITY_MAX: float = 140.0
    GRAD_CLIP:    float = 1.0
    WEIGHT_DECAY: float = 1e-4
    SEG_EPOCHS:   int   = 10                                 # U-Net lung seg (proper)
    # LRs scaled by √2 ≈ 1.41 relative to single-GPU baseline (batch 32→64)
    # ── Epoch schedule: proper full-training run ───────────────────────
    # S1: classification warmup  – 10 eps is sufficient, cls AUROC already 0.98
    # S2: full multi-task        – 15 eps to push severity Pearson r → 0.75+
    # S3: fine-tune + SWA        – 10 eps; SWA kicks in last 3 eps for stability
    S1_EPOCHS: int=10; S1_LR: float=4e-4
    S2_EPOCHS: int=15; S2_LR: float=4e-4; S2_REG_LR: float=4e-4
    S3_EPOCHS: int=10; S3_LR: float=1.4e-5; SWA_START: int=3
    WARMUP_EPOCHS: int=2; CYCLE_DECAY: float=0.5; ETA_MIN: float=1e-6
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP: bool = True
    # bfloat16: natively faster on T4 than fp16, no loss scaling needed,
    # and more stable for the regression head's small gradients.
    AMP_DTYPE: str = "bfloat16"

    def __post_init__(self):
        self.CKPT_DIR.mkdir(parents=True, exist_ok=True)

CFG = Config()
print(f"Device : {CFG.DEVICE} | GPUs: {CFG.NUM_GPUS} | AMP: {CFG.USE_AMP}")
print(f"Batch  : {CFG.BATCH_SIZE} ({CFG.BATCH_SIZE // max(CFG.NUM_GPUS,1)} per GPU) | Workers: {CFG.NUM_WORKERS}")
print(f"Image  : {CFG.IMAGE_SIZE}^2 | Folds: {CFG.N_FOLDS}")
print(f"Epochs : S1={CFG.S1_EPOCHS}  S2={CFG.S2_EPOCHS}  S3={CFG.S3_EPOCHS}  Seg={CFG.SEG_EPOCHS}")
print(f"LR     : S1={CFG.S1_LR}  S2={CFG.S2_LR}  S3={CFG.S3_LR}")


## Cell 2 — Dataset CSV Builder

In [ ]:
"""Cell 2 – Build unified labels.csv from all source datasets.
Uses rglob() for robust path discovery regardless of subdirectory nesting."""


def _find_images(root: Path, pattern: str = "*.png") -> List[Path]:
    """Recursively find all images matching pattern under root."""
    return sorted(root.rglob(pattern))


def _resolve_tbx11k_dir(cfg: "Config") -> Path:
    """Handle both standard and double-nested Kaggle dataset slug paths.

    Standard:  /kaggle/input/tbx11k-simplified/
    Nested:    /kaggle/input/datasets/vbookshelf/tbx11k-simplified/tbx11k-simplified/
    Also tries the datasets/<owner>/<slug>/<slug> pattern automatically.
    """
    base = cfg.TBX11K_DIR
    if base.exists():
        # Check if images/ or data.csv live directly here
        if (base / "images").exists() or (base / "data.csv").exists():
            return base
        # One more level in (e.g. tbx11k-simplified/ inside the slug root)
        for child in base.iterdir():
            if child.is_dir() and (child / "images").exists():
                return child

    # Try the /kaggle/input/datasets/... path variant
    alt = Path("/kaggle/input/datasets")
    if alt.exists():
        for candidate in alt.rglob("data.csv"):
            return candidate.parent

    return base  # fall back; _diagnose will warn


def _diagnose(cfg: "Config") -> None:
    tbx_dir = _resolve_tbx11k_dir(cfg)
    print("\n────────────────────────── Dataset path diagnostics ──────────────────────────────────────────")
    for name, d in [
        ("Shenzhen",   cfg.SHENZHEN_DIR),
        ("Shen masks", cfg.SHENZHEN_MASK_DIR),
        ("Montgomery", cfg.MONTGOMERY_DIR),
        ("TBX11K",     tbx_dir),
        ("Rahman",     cfg.RAHMAN_DIR),
    ]:
        exists = d.exists()
        count  = len(list(d.rglob("*.png"))) if exists else 0
        print(f"  {name:<12}: {'EXISTS' if exists else 'MISSING':7}  "
              f"({count} PNGs found)  →  {d}")
    print("────────────────────────────────────────────────────────────────────────────────────────────────\n")



def parse_shenzhen(cfg: "Config") -> pd.DataFrame:
    """Shenzhen CXR: filename suffix _0=normal, _1=TB.
    Works with flat or ChinaSet_AllFiles/CXR_png/ nesting."""
    rows = []

    # Find ALL PNGs under the dataset root that match Shenzhen naming
    all_pngs = [p for p in _find_images(cfg.SHENZHEN_DIR)
                if p.stem.startswith("CHNCXR")]

    if not all_pngs:
        print(f"  [WARN] No Shenzhen images found under {cfg.SHENZHEN_DIR}")
        return pd.DataFrame()

    # Build mask lookup: stem → mask path (from yoctoman/shcxr-lung-mask)
    mask_lookup: Dict[str, str] = {}
    if cfg.SHENZHEN_MASK_DIR.exists():
        for mp in _find_images(cfg.SHENZHEN_MASK_DIR):
            # mask filenames may be CHNCXR_0001_0_mask.png or CHNCXR_0001_0.png
            key = mp.stem.replace("_mask", "")
            mask_lookup[key] = str(mp)

    for p in all_pngs:
        stem     = p.stem                        # e.g. CHNCXR_0001_0
        tb_label = int(stem.split("_")[-1])      # 0 or 1
        mask_p   = mask_lookup.get(stem, "")
        rows.append(dict(
            image_path=str(p),
            patient_id=f"SZ_{stem}",
            source="shenzhen",
            tb_label=tb_label,
            has_severity=0,
            severity=-1.0,
            has_lung_mask=int(bool(mask_p)),
            lung_mask_path=mask_p,
        ))

    print(f"  Shenzhen   : {len(rows)} images  "
          f"({sum(r['tb_label'] for r in rows)} TB+, "
          f"{sum(r['has_lung_mask'] for r in rows)} masks)")
    return pd.DataFrame(rows)


def parse_montgomery(cfg: "Config") -> pd.DataFrame:
    """Montgomery CXR: L+R masks merged via logical OR.
    Works with flat or MontgomerySet/CXR_png/ nesting."""
    rows = []

    all_pngs = [p for p in _find_images(cfg.MONTGOMERY_DIR)
                if p.stem.startswith("MCUCXR") and "Mask" not in str(p)]

    if not all_pngs:
        print(f"  [WARN] No Montgomery images found under {cfg.MONTGOMERY_DIR}")
        return pd.DataFrame()

    # Build left/right mask lookups
    left_lookup:  Dict[str, Path] = {}
    right_lookup: Dict[str, Path] = {}
    for mp in _find_images(cfg.MONTGOMERY_DIR):
        if "leftMask" in str(mp)  and mp.stem.startswith("MCUCXR"):
            left_lookup[mp.stem]  = mp
        if "rightMask" in str(mp) and mp.stem.startswith("MCUCXR"):
            right_lookup[mp.stem] = mp

    mc_mask_dir = cfg.BASE / "mc_masks"
    mc_mask_dir.mkdir(exist_ok=True)

    for p in all_pngs:
        stem     = p.stem                        # e.g. MCUCXR_0001_0
        tb_label = int(stem.split("_")[-1])      # 0 or 1

        mask_p = ""
        lm = left_lookup.get(stem)
        rm = right_lookup.get(stem)
        if lm and rm:
            merged_path = mc_mask_dir / f"{stem}.png"
            if not merged_path.exists():
                l_arr = cv2.imread(str(lm), cv2.IMREAD_GRAYSCALE)
                r_arr = cv2.imread(str(rm), cv2.IMREAD_GRAYSCALE)
                if l_arr is not None and r_arr is not None:
                    if l_arr.shape != r_arr.shape:
                        r_arr = cv2.resize(r_arr, (l_arr.shape[1], l_arr.shape[0]))
                    merged = (np.logical_or(l_arr > 0, r_arr > 0).astype(np.uint8) * 255)
                    cv2.imwrite(str(merged_path), merged)
            if merged_path.exists():
                mask_p = str(merged_path)

        rows.append(dict(
            image_path=str(p),
            patient_id=f"MC_{stem}",
            source="montgomery",
            tb_label=tb_label,
            has_severity=0,
            severity=-1.0,
            has_lung_mask=int(bool(mask_p)),
            lung_mask_path=mask_p,
        ))

    print(f"  Montgomery : {len(rows)} images  "
          f"({sum(r['tb_label'] for r in rows)} TB+, "
          f"{sum(r['has_lung_mask'] for r in rows)} masks)")
    return pd.DataFrame(rows)


def parse_tbx11k(cfg: "Config") -> pd.DataFrame:
    """TBX11K-simplified: read data.csv for precise labels.

    CSV target column values:
      'active_tb'    → tb_label = 1
      'no_tb'        → tb_label = 0
      'latent_tb'    → tb_label = 0 (not infectious active TB)
      'sick_not_tb'  → tb_label = 0

    Falls back to filename prefix (tb*/h*/s*) if data.csv not found.
    Handles nested slug paths automatically via _resolve_tbx11k_dir().
    """
    rows = []
    tbx_dir = _resolve_tbx11k_dir(cfg)

    if not tbx_dir.exists():
        print(f"  [WARN] TBX11K dir not found: {tbx_dir}")
        return pd.DataFrame()

    print(f"  TBX11K root: {tbx_dir}")

    # Build image path lookup: fname → full path
    img_dir = tbx_dir / "images"
    all_imgs = list(img_dir.glob("*.png")) if img_dir.exists() \
               else list(tbx_dir.rglob("*.png"))
    img_lookup = {p.name: str(p) for p in all_imgs}

    if not img_lookup:
        print(f"  [WARN] No PNG images found under {tbx_dir}")
        return pd.DataFrame()

    # Try reading data.csv for precise labels
    csv_path = tbx_dir / "data.csv"
    if csv_path.exists():
        df_csv = pd.read_csv(csv_path)
        # One row per annotation (multiple rows for same image if multiple bboxes)
        # Deduplicate by fname keeping the first occurrence for label assignment
        df_unique = df_csv.drop_duplicates(subset=["fname"], keep="first")
        for _, row in df_unique.iterrows():
            fname    = str(row["fname"]).strip()
            img_path = img_lookup.get(fname)
            if not img_path:
                continue
            target   = str(row.get("target",  "")).strip().lower()
            tb_type  = str(row.get("tb_type", "")).strip().lower()
            # target column = 'tb' | 'no_tb'
            # tb_type column = 'active_tb' | 'latent_tb' | 'none'
            tb_label = 1 if tb_type == "active_tb" else 0
            rows.append(dict(
                image_path=img_path,
                patient_id=f"TBX_{Path(fname).stem.lower()}",
                source="tbx11k",
                tb_label=tb_label,
                has_severity=0,
                severity=-1.0,
                has_lung_mask=0,
                lung_mask_path="",
            ))
        print(f"  TBX11K     : {len(rows)} images (from data.csv)  "
              f"({sum(r['tb_label'] for r in rows)} active TB+)")
    else:
        # Fallback: filename-prefix heuristic
        print(f"  [WARN] data.csv not found, using filename prefix labels")
        for p in sorted(all_imgs):
            stem = p.stem.lower()
            tb_label = 1 if stem.startswith("tb") else 0
            rows.append(dict(
                image_path=str(p),
                patient_id=f"TBX_{stem}",
                source="tbx11k",
                tb_label=tb_label,
                has_severity=0,
                severity=-1.0,
                has_lung_mask=0,
                lung_mask_path="",
            ))
        print(f"  TBX11K     : {len(rows)} images (filename prefix)  "
              f"({sum(r['tb_label'] for r in rows)} TB+)")

    return pd.DataFrame(rows)



def parse_rahman(cfg: "Config") -> pd.DataFrame:
    """Optional Tawsifur Rahman dataset: folder-name labels."""
    rows = []
    if not cfg.RAHMAN_DIR.exists():
        return pd.DataFrame()

    for p in _find_images(cfg.RAHMAN_DIR):
        parts = p.parts
        if "Tuberculosis" in parts:
            tb_label = 1
        elif "Normal" in parts:
            tb_label = 0
        else:
            continue
        rows.append(dict(
            image_path=str(p),
            patient_id=f"RH_{p.stem}",
            source="rahman",
            tb_label=tb_label,
            has_severity=0,
            severity=-1.0,
            has_lung_mask=0,
            lung_mask_path="",
        ))

    if rows:
        print(f"  Rahman     : {len(rows)} images  "
              f"({sum(r['tb_label'] for r in rows)} TB+)")
    return pd.DataFrame(rows)


def build_labels_csv(cfg: "Config") -> pd.DataFrame:
    """Merge all datasets, drop duplicates, save labels.csv."""
    if cfg.LABELS_CSV.exists():
        print(f"Loading existing labels CSV: {cfg.LABELS_CSV}")
        df = pd.read_csv(cfg.LABELS_CSV)
        print(f"  {len(df)} rows | {df['tb_label'].sum()} TB+")
        return df

    _diagnose(cfg)
    print("Building labels.csv …")

    parts = [
        parse_shenzhen(cfg),
        parse_montgomery(cfg),
        parse_tbx11k(cfg),
        parse_rahman(cfg),
    ]

    valid = [p for p in parts if len(p) > 0]
    if not valid:
        raise RuntimeError(
            "No images found in any dataset!\n"
            "Make sure all 4 Kaggle dataset slugs are added via 'Add Data':\n"
            "  raddar/tuberculosis-chest-xrays-shenzhen\n"
            "  yoctoman/shcxr-lung-mask\n"
            "  raddar/tuberculosis-chest-xrays-montgomery\n"
            "  vbookshelf/tbx11k-simplified\n"
        )

    df = pd.concat(valid, ignore_index=True)
    df.drop_duplicates(subset=["image_path"], inplace=True)
    df.reset_index(drop=True, inplace=True)

    df.to_csv(cfg.LABELS_CSV, index=False)

    total  = len(df)
    tb_pos = df["tb_label"].sum()
    print(f"\n  Total  : {total} images")
    print(f"  TB+    : {int(tb_pos)}  ({100*tb_pos/total:.1f}%)")
    print(f"  Saved  → {cfg.LABELS_CSV}")
    return df


# Run
labels_df = build_labels_csv(CFG)
labels_df.head()


## Cell 3 — Lung Segmentation (U-Net)

In [ ]:
"""Cell 3 – Lung Segmentation: train U-Net on Montgomery+Shenzhen masks,
then apply to all images lacking a lung mask (mainly TBX11K)."""

# ── Dataset for lung segmenter training ──────────────────────────────
class LungMaskDataset(Dataset):
    def __init__(self, df: pd.DataFrame, size: int = 256):
        self.rows = df[df["has_lung_mask"] == 1].reset_index(drop=True)
        self.size = size
        self.tfm  = A.Compose([
            A.Resize(size, size),
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(0.1, 0.1, p=0.3),
            A.Normalize(mean=(0.485,), std=(0.229,)),
            ToTensorV2(),
        ])

    def __len__(self): return len(self.rows)

    def __getitem__(self, idx):
        r   = self.rows.iloc[idx]
        img = cv2.imread(r["image_path"], cv2.IMREAD_GRAYSCALE)
        msk = cv2.imread(r["lung_mask_path"], cv2.IMREAD_GRAYSCALE)
        # Guard against corrupted or missing files so the DataLoader never crashes
        if img is None:
            img = np.zeros((self.size, self.size), dtype=np.uint8)
        if msk is None:
            msk = np.zeros((self.size, self.size), dtype=np.uint8)
        img = np.stack([img, img, img], axis=-1)          # H×W×3
        msk = (msk > 127).astype(np.float32)
        aug  = self.tfm(image=img, mask=msk)
        return aug["image"], aug["mask"].unsqueeze(0)     # (3,H,W), (1,H,W)


def get_lung_unet() -> nn.Module:
    """U-Net with ResNet-34 ImageNet encoder from segmentation_models_pytorch."""
    return smp.Unet(
        encoder_name="resnet34",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
        activation=None,          # raw logits; we apply sigmoid at inference
    )


def train_lung_segmenter(df: pd.DataFrame, cfg: "Config") -> nn.Module:
    """Train lung U-Net on available masks (Montgomery + Shenzhen)."""
    if cfg.LUNG_CKPT.exists():
        print(f"Loading cached lung U-Net: {cfg.LUNG_CKPT}")
        model = get_lung_unet().to(cfg.DEVICE)
        model.load_state_dict(torch.load(cfg.LUNG_CKPT, map_location=cfg.DEVICE, weights_only=False))
        model.eval()
        return model

    print("Training lung segmentation U-Net …")
    ds     = LungMaskDataset(df, size=cfg.SEG_SIZE)
    if len(ds) == 0:
        print("  [WARN] No lung masks found for training (len(ds)==0). Skipping segmentation.")
        return None

    loader = DataLoader(ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                        num_workers=cfg.NUM_WORKERS, pin_memory=True)

    model = get_lung_unet().to(cfg.DEVICE)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)

    opt  = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    loss_fn = smp.losses.DiceLoss(mode="binary")
    scaler  = GradScaler("cuda") if cfg.USE_AMP else None

    best_loss  = float("inf")
    amp_dtype  = getattr(torch, cfg.AMP_DTYPE, torch.float16)  # honour cfg (bfloat16 or float16)
    for epoch in range(cfg.SEG_EPOCHS):
        model.train(); epoch_loss = 0.0
        for imgs, msks in loader:
            imgs, msks = imgs.to(cfg.DEVICE), msks.to(cfg.DEVICE)
            opt.zero_grad(set_to_none=True)
            if cfg.USE_AMP:
                with autocast("cuda", dtype=amp_dtype):
                    pred = model(imgs)
                    loss = loss_fn(pred, msks)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update()
            else:
                pred = model(imgs)
                loss = loss_fn(pred, msks)
                loss.backward(); opt.step()
            epoch_loss += loss.item()
        epoch_loss /= len(loader)
        if epoch % 5 == 0 or epoch == cfg.SEG_EPOCHS - 1:   # always print the last epoch
            print(f"  Seg epoch {epoch+1:02d}/{cfg.SEG_EPOCHS}  loss={epoch_loss:.4f}")
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            core = model.module if hasattr(model, "module") else model
            torch.save(core.state_dict(), cfg.LUNG_CKPT)

    print(f"Lung U-Net saved → {cfg.LUNG_CKPT}  (best Dice loss={best_loss:.4f})")
    core = model.module if hasattr(model, "module") else model
    core.eval()
    return core


@torch.no_grad()
def apply_lung_masks(df: pd.DataFrame, model: nn.Module, cfg: "Config") -> pd.DataFrame:
    """Predict lung masks for images that don't have one (e.g. TBX11K).
    Saves masks to /kaggle/working/predicted_masks/ and updates df."""
    out_dir = cfg.BASE / "predicted_masks"; out_dir.mkdir(exist_ok=True)
    needs   = df[df["has_lung_mask"] == 0].index.tolist()
    if not needs:
        print("All images already have lung masks.")
        return df

    if model is None:
        print("  [WARN] No lung model available. Skipping mask prediction.")
        return df

    print(f"Predicting lung masks for {len(needs)} images …")
    model.eval()
    tfm = A.Compose([
        A.Resize(cfg.SEG_SIZE, cfg.SEG_SIZE),
        A.Normalize(mean=(0.485,), std=(0.229,)),
        ToTensorV2(),
    ])

    for idx in tqdm(needs, desc="Lung masks"):
        img_path = df.at[idx, "image_path"]
        out_path = out_dir / (Path(img_path).stem + "_mask.png")
        if out_path.exists():
            df.at[idx, "lung_mask_path"] = str(out_path)
            df.at[idx, "has_lung_mask"]  = 1
            continue

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        h, w = img.shape
        img3 = np.stack([img, img, img], axis=-1)
        aug  = tfm(image=img3)
        inp  = aug["image"].unsqueeze(0).to(cfg.DEVICE)

        logit = model(inp)                                # (1,1,256,256)
        prob  = torch.sigmoid(logit).squeeze().cpu().numpy()
        msk   = (prob > 0.5).astype(np.uint8) * 255
        msk   = cv2.resize(msk, (w, h), interpolation=cv2.INTER_NEAREST)
        cv2.imwrite(str(out_path), msk)

        df.at[idx, "lung_mask_path"] = str(out_path)
        df.at[idx, "has_lung_mask"]  = 1

    df.to_csv(cfg.LABELS_CSV, index=False)
    print(f"Updated labels.csv with predicted masks.")
    return df


def crop_to_lung(img: np.ndarray, mask: np.ndarray,
                 target_size: int = 512, dilation_px: int = 10) -> np.ndarray:
    """Dilate lung mask, crop bounding box, pad to square, resize."""
    if mask.max() == 0:
        # fallback: centre-crop at 90% of image
        h, w = img.shape[:2]
        m = int(min(h, w) * 0.05)
        img = img[m:h-m, m:w-m]
    else:
        kern = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE, (dilation_px * 2 + 1, dilation_px * 2 + 1))
        dilated = cv2.dilate(mask, kern)
        ys, xs  = np.where(dilated > 0)
        y0, y1  = max(ys.min() - dilation_px, 0), min(ys.max() + dilation_px, img.shape[0])
        x0, x1  = max(xs.min() - dilation_px, 0), min(xs.max() + dilation_px, img.shape[1])
        img = img[y0:y1, x0:x1]

    # Pad to square
    h, w = img.shape[:2]
    side  = max(h, w)
    padded = np.zeros((side, side, 3) if img.ndim == 3 else (side, side), dtype=img.dtype)
    ph, pw = (side - h) // 2, (side - w) // 2
    if img.ndim == 3:
        padded[ph:ph+h, pw:pw+w, :] = img
    else:
        padded[ph:ph+h, pw:pw+w]    = img
    return cv2.resize(padded, (target_size, target_size))


# ── Run ───────────────────────────────────────────────────────────────
lung_model   = train_lung_segmenter(labels_df, CFG)
labels_df    = apply_lung_masks(labels_df, lung_model, CFG)
print(f"\nImages with lung masks: {labels_df['has_lung_mask'].sum()} / {len(labels_df)}")


## Cell 4 — Severity Labels (3-Tier Pipeline)

In [ ]:
"""Cell 4 – 3-Tier Severity Label Generation (Timika score 0-140).

Timika formula:  S = 100 * (A_lesion ∩ A_lung) / A_lung  +  40 * cavity_present

Tier 1 – Shenzhen/ChinaSet VIA polygon annotations  (Annotations_AllinOne_json.json)
         Using the consolidated all-in-one VGG Image Annotator (VIA) JSON.
Tier 2 – TBX11K COCO bounding boxes  →  pseudo-ALP + linear calibration
Tier 3 – Grad-CAM++ pseudo-labels for remaining unannotated TB+ cases
"""

import ast
import json as _json
from pathlib import Path

# ── Tier 1: ChinaSet / Shenzhen VIA polygon annotations ─────────────────
# Labels considered as active lesions (contributing to ALP)
LESION_LABELS = {
    "cavity",
    "cavitation",
    "small_infiltrate_(non-linear)",
    "moderate_infiltrate_(non-linear)",
    "severe_infiltrate_(consolidation)",
    "clustered_nodule_(2mm-5mm_apart)",
    "single_nodule_(non-calcified)",
    "linear_density",
    "apical_thickening",
    "pleural_thickening_(non-apical)",
    "thickening_of_interlobar_fissure",
    "retraction",
    "adenopathy",
    "other",
}

# Labels specifically indicating cavitation (adds +40 to Timika score)
CAVITY_LABELS = {"cavity", "cavitation"}


def _load_via_annotations(json_path: Path) -> dict:
    """Load the VIA all-in-one JSON and return a dict keyed by filename."""
    with open(json_path, "r") as f:
        raw = _json.load(f)

    # VIA format: keys are "<filename><size>", value has 'filename' and 'regions'
    # Build a clean mapping: filename -> list of regions
    fname_map: dict = {}
    for entry in raw.values():
        fname = entry.get("filename", "")
        regions = entry.get("regions", [])
        if fname:
            # Multiple entries with same filename are rare but union them
            fname_map.setdefault(fname, []).extend(regions)
    return fname_map


def _alp_from_via_regions(regions: list, lung_mask, orig_shape: tuple) -> tuple:
    """Compute ALP, cavity flag, and bbox-based ALP from VIA regions.

    Args:
        regions:    list of VIA region dicts (each with shape_attributes & region_attributes)
        lung_mask:  np.ndarray H×W grayscale lung segmentation mask
        orig_shape: (h_orig, w_orig) of the *original* CXR image the VIA annotations
                    were drawn on. Used to correctly scale polygon coordinates to
                    lung_mask dimensions. Must NOT be the polygon bounding box.

    Returns:
        (alp_percent, has_cavity, bbox_alp_percent):
            alp_percent   – polygon ALP in [0, 100] or -1 on error
            has_cavity     – bool
            bbox_alp_percent – ALP computed from bounding rectangles of the
                              polygons, used to calibrate Tier-2 bbox scores.
                              -1 on error.
    """
    h, w = lung_mask.shape[:2]
    lesion_mask = np.zeros((h, w), dtype=np.uint8)
    bbox_mask   = np.zeros((h, w), dtype=np.uint8)   # synthetic bbox mask
    has_cavity = False

    # Fix: use real original image dimensions for scaling instead of the
    # polygon bounding box. The old code used max(points_x) as the "original
    # width", which is wrong whenever the polygon doesn't reach the image
    # border — causing all coordinates to be compressed toward the origin.
    h_orig, w_orig = orig_shape
    sx = w / max(w_orig, 1)
    sy = h / max(h_orig, 1)

    for region in regions:
        shape = region.get("shape_attributes", {})
        attrs = region.get("region_attributes", {})

        # Determine the label from region_attributes (first key whose value == "good")
        label = ""
        for k, v in attrs.items():
            if v == "good":
                label = k.lower()
                break

        shape_name = shape.get("name", "")

        # ── Handle polygon and polyline shapes ──────────────────────────
        if shape_name in ("polygon", "polyline"):
            xs = shape.get("all_points_x", [])
            ys = shape.get("all_points_y", [])
            if len(xs) < 3 or len(ys) < 3:
                continue

            pts = np.array(
                [[int(x * sx), int(y * sy)] for x, y in zip(xs, ys)],
                dtype=np.int32,
            )
            pts = np.clip(pts, [0, 0], [w - 1, h - 1])

            # Only draw lesion polygons (skip non-lesion labels like calcified nodules)
            if label in LESION_LABELS or label == "":
                cv2.fillPoly(lesion_mask, [pts], 255)
                # Also fill the bounding rectangle of this polygon into bbox_mask
                bx, by, bw, bh = cv2.boundingRect(pts)
                bbox_mask[by:by + bh, bx:bx + bw] = 255

            if label in CAVITY_LABELS:
                has_cavity = True

        # ── Handle point markers (e.g. Pleural_Effusion point) ──────────
        elif shape_name == "point":
            # Points don't contribute area to ALP; only flag cavities if relevant
            if label in CAVITY_LABELS:
                has_cavity = True

    lung_area = (lung_mask > 127).sum()
    if lung_area == 0:
        return -1.0, False, -1.0

    lesion_in_lung = ((lesion_mask > 0) & (lung_mask > 127)).sum()
    alp = float(lesion_in_lung) / float(lung_area) * 100.0

    bbox_in_lung = ((bbox_mask > 0) & (lung_mask > 127)).sum()
    bbox_alp = float(bbox_in_lung) / float(lung_area) * 100.0

    return min(alp, 100.0), has_cavity, min(bbox_alp, 100.0)


def compute_tier1(df: pd.DataFrame, cfg: "Config") -> pd.DataFrame:
    """Add Tier-1 severity labels using the ChinaSet VIA all-in-one JSON.

    Looks for the annotation file at:
      1. cfg.SHEN_ANNOT_JSON  (if attribute exists)
      2. cfg.BASE / 'Annotations_AllinOne_json.json'
      3. The tb_model project root alongside this script
    """
    # Resolve annotation JSON path
    annot_json: Path | None = None
    candidates = []
    if hasattr(cfg, "SHEN_ANNOT_JSON"):
        candidates.append(Path(cfg.SHEN_ANNOT_JSON))
    candidates.append(Path(cfg.BASE) / "Annotations_AllinOne_json.json")

    for c in candidates:
        if c.exists():
            annot_json = c
            break

    if annot_json is None:
        print(
            "[Tier-1] Annotations_AllinOne_json.json not found in any candidate path. "
            "Tier-1 skipped; Tier-2/3 will be used."
        )
        print(f"  Searched: {[str(c) for c in candidates]}")
        return df

    print(f"[Tier-1] Loading VIA annotations from: {annot_json}")
    fname_map = _load_via_annotations(annot_json)
    print(f"  Loaded {len(fname_map)} annotated images.")

    updated = 0
    skipped_no_mask = 0
    skipped_no_annot = 0

    for idx, row in df[df["source"] == "shenzhen"].iterrows():
        if row["tb_label"] != 1:
            continue

        fname = Path(row["image_path"]).name
        regions = fname_map.get(fname)

        if regions is None:
            skipped_no_annot += 1
            continue

        if not row["has_lung_mask"]:
            skipped_no_mask += 1
            continue

        lung_mask = cv2.imread(str(row["lung_mask_path"]), cv2.IMREAD_GRAYSCALE)
        if lung_mask is None:
            skipped_no_mask += 1
            continue

        # Read original image dimensions for correct polygon coordinate scaling.
        # Fall back to lung_mask shape only if the image can't be read.
        orig_img = cv2.imread(str(row["image_path"]), cv2.IMREAD_GRAYSCALE)
        orig_shape = orig_img.shape[:2] if orig_img is not None else lung_mask.shape[:2]

        alp, cavity, bbox_alp = _alp_from_via_regions(regions, lung_mask, orig_shape=orig_shape)
        if alp < 0:
            continue

        timika = min(alp + 40.0 * float(cavity), 140.0)
        df.at[idx, "severity"]     = timika
        df.at[idx, "has_severity"] = 1
        # Store synthetic bbox ALP for Tier-2 calibration
        if bbox_alp >= 0:
            df.at[idx, "_raw_alp"] = bbox_alp
        updated += 1

    print(
        f"[Tier-1] Shenzhen VIA polygons: {updated} cases labelled "
        f"(skipped – no annotation: {skipped_no_annot}, no mask: {skipped_no_mask})"
    )
    return df


# ── Tier 2: TBX11K bounding-box pseudo-labels ────────────────────────
def compute_tier2(df: pd.DataFrame, cfg: "Config") -> pd.DataFrame:
    """Convert TBX11K data.csv bboxes → pseudo-ALP → Timika score.

    Bbox format in data.csv:  {'xmin': x, 'ymin': y, 'width': w, 'height': h}
    (Python dict stored as a string; parse with ast.literal_eval)

    Multiple bbox rows for the same image are unioned into one lesion mask.
    ALP = (lesion ∩ lung) / lung  *  100  → Timika = ALP (no cavity term for Tier-2).
    """
    # Resolve TBX11K root (same logic as parse_tbx11k)
    tbx_dir = _resolve_tbx11k_dir(cfg)
    csv_path = tbx_dir / "data.csv"

    if not csv_path.exists():
        print("[Tier-2] data.csv not found — skipping")
        return df

    df_csv = pd.read_csv(csv_path)
    # tb_type column = 'active_tb' | 'latent_tb' | 'none'
    df_tb  = df_csv[(df_csv["tb_type"] == "active_tb") &
                    (df_csv["bbox"].notna()) &
                    (df_csv["bbox"] != "none")].copy()

    if df_tb.empty:
        print("[Tier-2] No active TB bboxes in data.csv — skipping")
        return df

    # Group bboxes by filename (one image may have multiple boxes)
    fname2boxes: Dict[str, list] = {}
    for _, row in df_tb.iterrows():
        fname = str(row["fname"]).strip()
        try:
            b = ast.literal_eval(str(row["bbox"]))
            fname2boxes.setdefault(fname, []).append(b)
        except Exception:
            continue

    updated = 0

    # ── Gather calibration pairs from Tier-1 synthetic bbox ALP ───────
    # Tier-1 stored _raw_alp (bbox-based ALP) alongside the gold-standard
    # Timika severity.  We use these paired values to compute the dynamic
    # calibration slope *before* processing any TBX11K rows.
    tier1_alp, raw_alp_paired = [], []
    if "_raw_alp" in df.columns:
        calib_mask = (df["has_severity"] == 1) & (df["_raw_alp"].notna()) & (df["_raw_alp"] >= 0)
        for _, row in df[calib_mask].iterrows():
            raw_alp_paired.append(float(row["_raw_alp"]))
            tier1_alp.append(float(row["severity"]))
        if tier1_alp:
            print(f"  [Tier-2] Found {len(tier1_alp)} Tier-1 calibration pairs "
                  f"(bbox ALP vs gold-standard Timika)")

    for idx, row in df[df["source"] == "tbx11k"].iterrows():
        if row["tb_label"] != 1:
            continue
        fname = Path(row["image_path"]).name
        boxes = fname2boxes.get(fname, [])
        if not boxes:
            continue

        # ── Compute ALP ──────────────────────────────────────────────
        if row["has_lung_mask"] and row["lung_mask_path"]:
            # Preferred: lung-relative ALP
            lung_mask = cv2.imread(str(row["lung_mask_path"]), cv2.IMREAD_GRAYSCALE)
            if lung_mask is not None:
                h, w      = lung_mask.shape
                lung_area = (lung_mask > 127).sum()
                pseudo    = np.zeros((h, w), dtype=np.uint8)
                for b in boxes:
                    sx, sy = w / 512.0, h / 512.0
                    x0 = max(0, int(b["xmin"] * sx))
                    y0 = max(0, int(b["ymin"] * sy))
                    x1 = min(w - 1, int((b["xmin"] + b["width"])  * sx))
                    y1 = min(h - 1, int((b["ymin"] + b["height"]) * sy))
                    pseudo[y0:y1, x0:x1] = 255
                if lung_area == 0:
                    continue
                overlap = ((pseudo > 0) & (lung_mask > 127)).sum()
                raw_alp = float(overlap) / float(lung_area) * 100.0
            else:
                lung_mask = None
        else:
            lung_mask = None

        if lung_mask is None:
            # Fix: skip images without a lung mask entirely rather than using
            # a flat 60%-of-image fallback.  That heuristic introduced a
            # systematic downward ALP bias (compressed/pathological lungs may
            # occupy far less than 60%) and corrupted the regression targets.
            # These cases stay has_severity=0 and are handled by Tier-3 if
            # they have a predicted mask, or excluded from severity training.
            continue

        df.at[idx, "_raw_alp"] = raw_alp
        updated += 1

    if not updated:
        print("[Tier-2] No TBX11K active-TB rows matched bboxes in data.csv — skipping")
        return df

    # Linear calibration against Tier-1 synthetic bbox pairs
    slope = 1.5  # paper default fallback
    if len(tier1_alp) >= 5:
        X = np.array(raw_alp_paired).reshape(-1, 1)
        y = np.array(tier1_alp)
        # fit_intercept=False: 0% bbox area must map to 0 severity
        reg = LinearRegression(fit_intercept=False).fit(X, y)
        slope = float(np.clip(reg.coef_[0], 1.0, 2.5))
        print(f"  [Tier-2] Dynamic calibration slope = {slope:.3f} "
              f"(from {len(tier1_alp)} Tier-1 pairs)")
    else:
        print(f"  [Tier-2] Only {len(tier1_alp)} Tier-1 pairs found "
              f"(need ≥5) — using default slope={slope}")

    for idx in df[df["source"] == "tbx11k"].index:
        raw = df.at[idx, "_raw_alp"] if "_raw_alp" in df.columns else -1
        if raw < 0:
            continue
        timika = min(raw * slope, 100.0)   # no cavity bonus in data.csv
        df.at[idx, "severity"]     = timika
        df.at[idx, "has_severity"] = 1

    df.drop(columns=["_raw_alp"], errors="ignore", inplace=True)
    print(f"[Tier-2] TBX11K data.csv bbox: {updated} cases labelled  (slope={slope:.2f})")
    return df


# ── Tier 3: Grad-CAM++ pseudo-labels ─────────────────────────────────
from pytorch_grad_cam import GradCAMPlusPlus
# Note: targets=None is used below (not ClassifierOutputTarget) — correct for
# a single-output sigmoid binary model where we want TB-positive gradients.


def _train_teacher(df: pd.DataFrame, cfg: "Config") -> nn.Module:
    """Train a lightweight DenseNet-121 teacher on Tier-1+Tier-2 labelled data."""
    # Cache key includes df length so a stale teacher (trained on the full dataset
    # before the circular-loop fix) is never accidentally reloaded.
    ckpt = cfg.CKPT_DIR / f"teacher_densenet_{len(df)}imgs.pt"
    model = timm.create_model("densenet121", pretrained=True, num_classes=1)
    model = model.to(cfg.DEVICE)
    if ckpt.exists():
        state_dict = torch.load(ckpt, map_location=cfg.DEVICE, weights_only=False)
        state_dict = {k[7:] if k.startswith("module.") else k: v for k, v in state_dict.items()}
        model.load_state_dict(state_dict)
        model.eval(); return model

    # Simple dataset of all images with known tb_label
    class SimpleDS(Dataset):
        def __init__(self, df):
            self.df  = df.reset_index(drop=True)
            self.tfm = A.Compose([
                A.Resize(224, 224),
                A.HorizontalFlip(p=0.5),
                A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
                ToTensorV2(),
            ])
        def __len__(self): return len(self.df)
        def __getitem__(self, i):
            r   = self.df.iloc[i]
            img = cv2.imread(str(r["image_path"]), cv2.IMREAD_GRAYSCALE)
            if img is None: img = np.zeros((224, 224), dtype=np.uint8)
            img = np.stack([img] * 3, axis=-1)
            aug = self.tfm(image=img)
            lbl = torch.tensor([float(r["tb_label"])])
            return aug["image"], lbl

    ds = SimpleDS(df)
    loader = DataLoader(ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                        num_workers=cfg.NUM_WORKERS, pin_memory=True)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    opt    = torch.optim.Adam(model.parameters(), lr=1e-4)
    pw     = torch.tensor([5.95]).to(cfg.DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)
    for ep in range(10):
        model.train(); total = 0.0
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(cfg.DEVICE), lbls.to(cfg.DEVICE)
            opt.zero_grad()
            out  = model(imgs)
            loss = loss_fn(out, lbls)
            loss.backward(); opt.step()
            total += loss.item()
        if ep % 3 == 0:
            print(f"  Teacher epoch {ep+1}/10  loss={total/len(loader):.4f}")
    state_to_save = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    torch.save(state_to_save, ckpt)
    model.eval(); return model


def compute_tier3(df: pd.DataFrame, cfg: "Config") -> pd.DataFrame:
    """Generate Grad-CAM++ pseudo severity labels for unannotated TB+ cases."""
    needs = df[
        (df["tb_label"] == 1) &
        (df["has_severity"] == 0) &
        (df["has_lung_mask"] == 1)
    ].index.tolist()
    if not needs:
        print("[Tier-3] No unannotated TB+ cases to pseudo-label")
        return df

    print(f"[Tier-3] Pseudo-labelling {len(needs)} unannotated TB+ cases …")

    # ── Fix: break the circular pseudo-label loop ─────────────────────
    # Train the teacher ONLY on images it will NOT pseudo-label.
    # Tier-3 candidates (needs) are excluded so the teacher never memorises
    # the very images it will generate labels for.
    teacher_df = df.drop(index=needs).reset_index(drop=True)
    print(f"  Teacher will train on {len(teacher_df)} images "
          f"(excluded {len(needs)} Tier-3 candidates from training).")
    teacher = _train_teacher(teacher_df, cfg)
    teacher.eval()

    # GradCAM++ targets the last denseblock.
    # Unwrap DataParallel so .features is accessible on the underlying module.
    _teacher_core = teacher.module if isinstance(teacher, nn.DataParallel) else teacher
    target_layer = [_teacher_core.features.denseblock4.denselayer16.conv2]
    cam = GradCAMPlusPlus(model=teacher, target_layers=target_layer)

    tfm_val = A.Compose([
        A.Resize(224, 224),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

    updated = 0
    for idx in tqdm(needs, desc="GradCAM++ pseudo-labels"):
        row      = df.iloc[idx]
        img_raw  = cv2.imread(str(row["image_path"]), cv2.IMREAD_GRAYSCALE)
        mask_raw = cv2.imread(str(row["lung_mask_path"]), cv2.IMREAD_GRAYSCALE)
        if img_raw is None or mask_raw is None:
            continue

        img3 = np.stack([img_raw] * 3, axis=-1)
        aug  = tfm_val(image=img3)
        inp  = aug["image"].unsqueeze(0).to(cfg.DEVICE)

        # Fix: use targets=None so GradCAM++ activates gradients toward the
        # model's sole output (TB probability) in the *positive* direction.
        # ClassifierOutputTarget(0) was wrong — it targeted class index 0 which,
        # for a 1-output sigmoid, pushes gradients toward "not TB".
        grayscale_cam = cam(
            input_tensor=inp,
            targets=None
        )[0]  # (224, 224)

        # Resize back to lung-mask size
        h, w = mask_raw.shape
        cam_resized = cv2.resize(grayscale_cam, (w, h))

        # Threshold at per-image quantile within lung mask
        lung_px = cam_resized[mask_raw > 127]
        if len(lung_px) == 0:
            continue
        thresh = np.quantile(lung_px, cfg.GRADCAM_Q if hasattr(cfg, "GRADCAM_Q") else 0.4)
        lesion = ((cam_resized >= thresh) & (mask_raw > 127)).astype(np.uint8)

        # Morphological refinement
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        lesion = cv2.morphologyEx(lesion, cv2.MORPH_CLOSE, kernel)
        lesion = cv2.morphologyEx(lesion, cv2.MORPH_OPEN,  kernel)

        lung_area = (mask_raw > 127).sum()
        if lung_area == 0:
            continue
        alp    = float(lesion.sum()) / float(lung_area) * 100.0
        alp    = min(alp, 100.0)
        # No cavity information → cavity = 0 for Tier-3 (conservative)
        timika = float(alp)
        df.at[idx, "severity"]     = timika
        df.at[idx, "has_severity"] = 1
        updated += 1

    cam.__exit__(None, None, None)   # release forward/backward hooks properly
    print(f"[Tier-3] GradCAM++: {updated} cases labelled")
    return df


def add_severity_quartile(df: pd.DataFrame) -> pd.DataFrame:
    """Add stratification key for StratifiedGroupKFold."""
    df["sev_quartile"] = -1
    mask = df["has_severity"] == 1
    df.loc[mask, "sev_quartile"] = pd.qcut(
        df.loc[mask, "severity"], q=4, labels=False, duplicates="drop"
    )
    # Combined stratification key: tb_label * 10 + sev_quartile+1
    df["strat_key"] = (
        df["tb_label"].astype(str) + "_" + df["sev_quartile"].astype(str)
    )
    return df


# ── Run severity pipeline ─────────────────────────────────────────────
if not (labels_df["has_severity"] == 1).any():
    labels_df = compute_tier1(labels_df, CFG)
    labels_df = compute_tier2(labels_df, CFG)
    labels_df = compute_tier3(labels_df, CFG)
    labels_df = add_severity_quartile(labels_df)
    labels_df.to_csv(CFG.LABELS_CSV, index=False)
    print(
        f"\nSeverity coverage: {labels_df['has_severity'].sum()} / "
        f"{labels_df['tb_label'].sum()} TB+ cases"
    )
else:
    print("Severity labels already present — skipping generation.")
    labels_df = add_severity_quartile(labels_df)
    labels_df.to_csv(CFG.LABELS_CSV, index=False)  # persist strat_key so next run loads it


## Cell 5 — PyTorch Dataset + Transforms

In [ ]:
"""Cell 5 – PyTorch Dataset + Albumentations transforms."""

def get_transforms(cfg: "Config", train: bool = True) -> A.Compose:
    if train:
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=10, p=0.5),
            A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.5),
            A.RandomBrightnessContrast(0.1, 0.1, p=0.3),
            A.ElasticTransform(alpha=120, sigma=6, p=0.3),
            A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.25),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])


class TBXDataset(Dataset):
    """
    Returns dict with keys:
      image        : (3, H, W) float32 tensor
      tb_label     : (1,) float32
      severity     : (1,) float32  — normalised to [0,1] (÷140)
      severity_mask: (1,) float32  — 1 if severity is valid, else 0
      patient_id   : str
    """
    def __init__(self, df: pd.DataFrame, cfg: "Config", train: bool = True):
        self.df    = df.reset_index(drop=True)
        self.cfg   = cfg
        self.train = train
        self.tfm   = get_transforms(cfg, train)

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, path: str) -> np.ndarray:
        """Load grayscale CXR → per-image min-max stretch → 3-channel uint8 array."""
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.zeros((self.cfg.IMAGE_SIZE, self.cfg.IMAGE_SIZE), dtype=np.uint8)
        # Per-image min-max normalisation: stretches each CXR to fill the full
        # [0, 255] range regardless of scanner brightness/contrast offset.
        # The previous code (÷255 then ×255) was a mathematical no-op.
        img_f = img.astype(np.float32)
        lo, hi = img_f.min(), img_f.max()
        img = ((img_f - lo) / (hi - lo + 1e-6) * 255.0).clip(0, 255).astype(np.uint8)
        img = np.stack([img, img, img], axis=-1)
        return img                              # (H, W, 3) uint8

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]

        # Load and crop to lung
        img  = self._load_image(row["image_path"])
        if row["has_lung_mask"] and row["lung_mask_path"]:
            msk = cv2.imread(row["lung_mask_path"], cv2.IMREAD_GRAYSCALE)
            if msk is not None:
                img = crop_to_lung(img, msk, self.cfg.IMAGE_SIZE)
        img = cv2.resize(img, (self.cfg.IMAGE_SIZE, self.cfg.IMAGE_SIZE))

        # Albumentations
        aug   = self.tfm(image=img)
        image = aug["image"]                   # (3, H, W)

        # Labels
        tb_label = torch.tensor([float(row["tb_label"])], dtype=torch.float32)

        has_sev       = bool(row["has_severity"]) and float(row["severity"]) >= 0
        severity_raw  = float(row["severity"]) if has_sev else 0.0
        severity_norm = severity_raw / self.cfg.SEVERITY_MAX   # [0, 1]
        severity      = torch.tensor([severity_norm], dtype=torch.float32)
        severity_mask = torch.tensor([float(has_sev)], dtype=torch.float32)

        return {
            "image":         image,
            "tb_label":      tb_label,
            "severity":      severity,
            "severity_mask": severity_mask,
            "patient_id":    str(row["patient_id"]),
        }


def make_loaders(
    df: pd.DataFrame,
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    cfg: "Config",
) -> Tuple[DataLoader, DataLoader]:
    train_ds = TBXDataset(df.iloc[train_idx], cfg, train=True)
    val_ds   = TBXDataset(df.iloc[val_idx],   cfg, train=False)
    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4,          # pre-load more batches; feeds GPU during CLAHE+Elastic
        drop_last=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.BATCH_SIZE,
        shuffle=False,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4,
    )
    return train_loader, val_loader


# ── Smoke-test ────────────────────────────────────────────────────────
_sample_ds = TBXDataset(labels_df.head(8), CFG, train=True)
_batch = _sample_ds[0]
print("Image shape     :", _batch["image"].shape)
print("tb_label        :", _batch["tb_label"])
print("severity (norm) :", _batch["severity"])
print("severity_mask   :", _batch["severity_mask"])


## Cell 6 — TB-MTNet Architecture + Loss

In [ ]:
"""Cell 6 – TB-MTNet architecture + MultiTaskLoss (Kendall–Gal–Cipolla)."""

# ── ECA Channel Attention ─────────────────────────────────────────────
class ECA(nn.Module):
    """Efficient Channel Attention (Wang et al., CVPR 2020). ~5 params."""
    def __init__(self, channels: int, k: int = 5):
        super().__init__()
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=k // 2, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, H, W)
        y = self.gap(x).squeeze(-1).transpose(-1, -2)   # (B, 1, C)
        y = self.conv(y).transpose(-1, -2).unsqueeze(-1) # (B, C, 1, 1)
        return x * torch.sigmoid(y)


# ── Projection Bridge: 2048 → 512 → 96 ───────────────────────────────
class ProjectionBridge(nn.Module):
    """2048→512→96 two-stage projection (paper §5.1)."""
    def __init__(self, in_ch: int = 2048, mid_ch: int = 512, out_ch: int = 96):
        super().__init__()
        self.bridge = nn.Sequential(
            nn.Conv2d(in_ch,  mid_ch, 1, bias=False), nn.BatchNorm2d(mid_ch), nn.GELU(),
            nn.Conv2d(mid_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch), nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.bridge(x)   # (B, 96, 14, 14)


# ── TB-MTNet ─────────────────────────────────────────────────────────
class TBMTNet(nn.Module):
    """
    Forward path (paper §5.1):
      Input (B,3,512,512)
      → Inception-v3 features_only → (B, 2048, 14, 14)
      → ECA(k=5)
      → ProjectionBridge 2048→512→96 → (B, 96, 14, 14)
      → flatten → (B, 196, 96)  +  [CLS]  +  pos-embed  → (B, 197, 96)
      → 4-layer pre-norm Transformer (d=96, h=4, ffn=384, GELU)
      → LayerNorm → CLS token → (B, 96)
      → cls head: Linear(96→1) sigmoid   [TB probability]
      → reg head: Linear(96→1) sigmoid   [Timika / 140]
    """
    def __init__(self, cfg: "Config"):
        super().__init__()
        d = cfg.D_MODEL

        # Backbone
        self.backbone = timm.create_model(
            "inception_v3",
            pretrained=True,
            features_only=True,
            out_indices=(4,),
        )
        self.eca    = ECA(channels=2048, k=5)
        self.bridge = ProjectionBridge(2048, 512, d)

        # Transformer
        self.cls_token  = nn.Parameter(torch.zeros(1, 1, d))
        self.pos_embed  = nn.Parameter(torch.zeros(1, 197, d))  # 196 patches + 1 CLS
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d,
            nhead=cfg.N_HEADS,
            dim_feedforward=cfg.FFN_DIM,
            dropout=cfg.DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True,     # pre-norm (paper)
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=cfg.N_LAYERS)
        self.norm        = nn.LayerNorm(d)

        # Heads
        self.cls_head = nn.Linear(d, 1)   # TB detection (logit → BCEWithLogits)
        self.reg_head = nn.Linear(d, 1)   # Timika/140  (sigmoid output)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # Backbone
        feats = self.backbone(x)[0]         # (B, 2048, 14, 14)
        feats = self.eca(feats)
        feats = self.bridge(feats)          # (B, 96, 14, 14)

        # Flatten spatial → tokens
        B, C, H, W = feats.shape
        tokens = feats.flatten(2).transpose(1, 2)   # (B, 196, 96)

        # Prepend CLS token
        cls   = self.cls_token.expand(B, -1, -1)    # (B, 1, 96)
        tokens = torch.cat([cls, tokens], dim=1)     # (B, 197, 96)
        tokens = tokens + self.pos_embed

        # Transformer
        out = self.transformer(tokens)   # (B, 197, 96)
        out = self.norm(out)
        cls_out = out[:, 0]              # (B, 96)

        # Heads
        logit  = self.cls_head(cls_out)                # (B, 1)  — raw logit
        sev    = torch.sigmoid(self.reg_head(cls_out)) # (B, 1)  — in [0, 1]
        return logit, sev

    def predict(self, x: torch.Tensor):
        """Returns (tb_prob, timika_score) for inference."""
        logit, sev = self.forward(x)
        return torch.sigmoid(logit), sev * 140.0


# ── Pearson Correlation Loss ──────────────────────────────────────────
def pearson_loss(pred: torch.Tensor, target: torch.Tensor,
                mask: torch.Tensor) -> torch.Tensor:
    """1 − Pearson r, computed only over masked (TB-positive) samples.
    Directly optimises the ranking metric the evaluation cares about."""
    pred_m   = pred[mask.bool()]
    target_m = target[mask.bool()]
    if len(pred_m) < 2:
        return torch.tensor(0.0, device=pred.device)
    vx  = pred_m   - pred_m.mean()
    vy  = target_m - target_m.mean()
    corr = (vx * vy).sum() / (
        (vx.norm() * vy.norm()).clamp(min=1e-8))
    return 1.0 - corr


# ── Multi-Task Loss (Kendall–Gal–Cipolla uncertainty weighting) ───────
class MultiTaskLoss(nn.Module):
    """
    L = exp(-s_c) * L_BCE + exp(-s_r) * (L_Huber + 0.5·L_Pearson) + s_c + s_r
    s_c, s_r are learnable log-variance parameters.
    Regression loss is masked to TB-positive cases only.
    """
    def __init__(self, pos_weight: float, huber_beta: float):
        super().__init__()
        self.s_c = nn.Parameter(torch.zeros(()))          # log-var for cls
        self.s_r = nn.Parameter(torch.tensor(-1.0))       # start high to prioritise severity
        pw = torch.tensor([pos_weight])
        self.bce   = nn.BCEWithLogitsLoss(pos_weight=pw)
        self.huber = nn.SmoothL1Loss(beta=huber_beta, reduction="none")

    def forward(
        self,
        logit: torch.Tensor,        # (B, 1) raw logit
        y_cls: torch.Tensor,        # (B, 1) binary TB label
        sev_pred: torch.Tensor,     # (B, 1) sigmoid output [0, 1]
        y_sev: torch.Tensor,        # (B, 1) normalised severity [0, 1]
        sev_mask: torch.Tensor,     # (B, 1) 1 if severity valid
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:

        l_cls = self.bce(logit, y_cls)

        # Regression: Huber (absolute distance) + Pearson (ranking)
        huber_elem = self.huber(sev_pred, y_sev)        # (B, 1)
        n_valid    = sev_mask.sum().clamp(min=1.0)
        l_huber    = (huber_elem * sev_mask).sum() / n_valid
        l_pearson  = pearson_loss(sev_pred, y_sev, sev_mask)
        l_reg      = l_huber + 0.5 * l_pearson

        loss = (torch.exp(-self.s_c) * l_cls + self.s_c +
                torch.exp(-self.s_r) * l_reg  + self.s_r)
        return loss, l_cls, l_reg


# ── Utilities ─────────────────────────────────────────────────────────
def count_parameters(model: nn.Module) -> int:
    total  = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total params     : {total/1e6:.2f}M")
    print(f"Trainable params : {trainable/1e6:.2f}M")
    return total


# ── Instantiate & sanity-check ────────────────────────────────────────
model    = TBMTNet(CFG).to(CFG.DEVICE)
mtl_loss = MultiTaskLoss(CFG.POS_WEIGHT, CFG.HUBER_BETA).to(CFG.DEVICE)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print(f"Using DataParallel across {torch.cuda.device_count()} GPUs")

count_parameters(model)

# Forward-pass smoke test
_x = torch.randn(2, 3, CFG.IMAGE_SIZE, CFG.IMAGE_SIZE).to(CFG.DEVICE)
with torch.no_grad():
    _core = model.module if hasattr(model, "module") else model
    _logit, _sev = _core(_x)
print(f"Logit shape : {_logit.shape}  |  Sev shape : {_sev.shape}")
assert _logit.shape == (2, 1) and _sev.shape == (2, 1), "Shape mismatch!"
print("Architecture OK ✓")


## Cell 7 — 3-Stage Training (5-Fold CV)

In [ ]:
"""Cell 7 – 3-Stage Progressive Training with 5-Fold Cross-Validation."""

from torch.optim.swa_utils import AveragedModel, SWALR

# ── Layer-wise LR decay for Inception-v3 backbone ────────────────────
def _layerwise_backbone_params(backbone: nn.Module,
                                base_lr: float,
                                decay: float = 0.8) -> list:
    """Return AdamW-style param-group dicts with depth-aware LR scaling.

    Inception-v3 stage ordering (shallow → deep):
      Stage 0 (earliest) : Conv2d_1a … Conv2d_4a  →  LR × decay³  (≈ 0.51×)
      Stage 1 (mid)      : Mixed_5b … Mixed_6b    →  LR × decay²  (≈ 0.64×)
      Stage 2 (late-mid) : Mixed_6c … Mixed_6e    →  LR × decay¹  (≈ 0.80×)
      Stage 3 (deepest)  : Mixed_7a … Mixed_7c    →  LR × decay⁰  (= 1.00×)

    Any unmatched parameters (e.g. timm feature_info wrappers) fall back
    to base_lr so nothing is silently excluded from training.
    """
    stage_prefixes = [
        # stage 0 – earliest, most generic features
        ["Conv2d_1a_3x3", "Conv2d_2a_3x3", "Conv2d_2b_3x3",
         "Conv2d_3b_1x1", "Conv2d_4a_3x3"],
        # stage 1
        ["Mixed_5b", "Mixed_5c", "Mixed_5d", "Mixed_6a", "Mixed_6b"],
        # stage 2
        ["Mixed_6c", "Mixed_6d", "Mixed_6e"],
        # stage 3 – deepest / most task-specific
        ["Mixed_7a", "Mixed_7b", "Mixed_7c"],
    ]
    n_stages   = len(stage_prefixes)
    assigned   = set()
    param_groups = []

    for stage_idx, prefixes in enumerate(stage_prefixes):
        lr_scale    = decay ** (n_stages - 1 - stage_idx)   # deeper → higher LR
        stage_params = []
        for name, param in backbone.named_parameters():
            if name not in assigned and any(name.startswith(pf) for pf in prefixes):
                stage_params.append(param)
                assigned.add(name)
        if stage_params:
            param_groups.append({"params": stage_params, "lr": base_lr * lr_scale})

    # Safety net: any unmatched params get base_lr
    remaining = [p for n, p in backbone.named_parameters() if n not in assigned]
    if remaining:
        param_groups.append({"params": remaining, "lr": base_lr})

    return param_groups


def make_optimizer(model: nn.Module, cfg: "Config",
                   stage: int, mtl_loss: nn.Module) -> torch.optim.Optimizer:
    core = model.module if hasattr(model, "module") else model
    if stage == 1:
        params = [
            # Backbone: layer-wise decay — early layers learn slower
            *_layerwise_backbone_params(core.backbone, cfg.S1_LR),
            {"params": core.eca.parameters(),        "lr": cfg.S1_LR},
            {"params": core.bridge.parameters(),     "lr": cfg.S1_LR},
            {"params": core.transformer.parameters(),"lr": cfg.S1_LR},
            {"params": core.norm.parameters(),       "lr": cfg.S1_LR},
            {"params": core.cls_head.parameters(),   "lr": cfg.S1_LR},
            {"params": core.cls_token,               "lr": cfg.S1_LR},
            {"params": core.pos_embed,               "lr": cfg.S1_LR},
            {"params": mtl_loss.parameters(),        "lr": cfg.S1_LR},
        ]
    elif stage == 2:
        params = [
            # Backbone: layer-wise decay — early layers learn slower
            *_layerwise_backbone_params(core.backbone, cfg.S2_LR),
            {"params": core.eca.parameters(),         "lr": cfg.S2_LR},
            {"params": core.bridge.parameters(),      "lr": cfg.S2_LR},
            {"params": core.transformer.parameters(), "lr": cfg.S2_LR},
            {"params": core.norm.parameters(),        "lr": cfg.S2_LR},
            {"params": core.cls_head.parameters(),    "lr": cfg.S2_LR},
            {"params": core.reg_head.parameters(),    "lr": cfg.S2_REG_LR},
            {"params": core.cls_token,                "lr": cfg.S2_LR},
            {"params": core.pos_embed,                "lr": cfg.S2_LR},
            {"params": mtl_loss.parameters(),         "lr": cfg.S2_LR},
        ]
    else:  # stage 3
        params = core.parameters()   # use core to avoid DataParallel duplicate-param warnings
        return torch.optim.AdamW(params, lr=cfg.S3_LR,
                                 weight_decay=cfg.WEIGHT_DECAY)
    return torch.optim.AdamW(params, weight_decay=cfg.WEIGHT_DECAY)


def make_scheduler(opt, n_epochs: int, cfg: "Config") -> CosineLRScheduler:
    return CosineLRScheduler(
        opt,
        t_initial=n_epochs,
        lr_min=cfg.ETA_MIN,
        warmup_t=cfg.WARMUP_EPOCHS,
        warmup_lr_init=1e-5,
        cycle_decay=cfg.CYCLE_DECAY,
    )


def train_epoch(model, loader, opt, scheduler, mtl_loss, scaler,
                cfg, epoch, freeze_reg=False):
    model.train()
    core = model.module if hasattr(model, "module") else model
    if freeze_reg:
        for p in core.reg_head.parameters():
            p.requires_grad_(False)
    else:
        for p in core.reg_head.parameters():
            p.requires_grad_(True)

    total_loss = cls_loss_sum = reg_loss_sum = 0.0
    for batch in loader:
        imgs  = batch["image"].to(cfg.DEVICE)
        y_cls = batch["tb_label"].to(cfg.DEVICE)
        y_sev = batch["severity"].to(cfg.DEVICE)
        s_msk = batch["severity_mask"].to(cfg.DEVICE)

        opt.zero_grad(set_to_none=True)
        amp_dtype = getattr(torch, cfg.AMP_DTYPE)  # bfloat16 or float16
        use_scaler = scaler is not None and cfg.AMP_DTYPE == "float16"
        if cfg.USE_AMP:
            with autocast("cuda", dtype=amp_dtype):
                logit, sev = model(imgs)
                loss, l_c, l_r = mtl_loss(logit, y_cls, sev, y_sev, s_msk)
            if use_scaler:
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
                scaler.step(opt); scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
                opt.step()
        else:
            logit, sev = model(imgs)       # DataParallel splits batch across GPUs
            loss, l_c, l_r = mtl_loss(logit, y_cls, sev, y_sev, s_msk)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            opt.step()

        total_loss   += loss.item()
        cls_loss_sum += l_c.item()
        reg_loss_sum += l_r.item()

    n = len(loader)
    scheduler.step(epoch + 1)
    return total_loss/n, cls_loss_sum/n, reg_loss_sum/n


@torch.no_grad()
def validate(model, loader, mtl_loss, cfg):
    model.eval()
    core = model.module if hasattr(model, "module") else model
    all_probs, all_labels, all_sev_pred, all_sev_true, all_sev_mask = [], [], [], [], []
    total_loss = 0.0

    for batch in loader:
        imgs  = batch["image"].to(cfg.DEVICE)
        y_cls = batch["tb_label"].to(cfg.DEVICE)
        y_sev = batch["severity"].to(cfg.DEVICE)
        s_msk = batch["severity_mask"].to(cfg.DEVICE)

        amp_dtype = getattr(torch, cfg.AMP_DTYPE)
        if cfg.USE_AMP:
            with autocast("cuda", dtype=amp_dtype):
                logit, sev = model(imgs)
                loss, _, _ = mtl_loss(logit, y_cls, sev, y_sev, s_msk)
        else:
            logit, sev = model(imgs)
            loss, _, _ = mtl_loss(logit, y_cls, sev, y_sev, s_msk)

        total_loss += loss.item()
        all_probs.append(torch.sigmoid(logit).cpu())
        all_labels.append(y_cls.cpu())
        all_sev_pred.append(sev.cpu() * cfg.SEVERITY_MAX)
        all_sev_true.append(y_sev.cpu() * cfg.SEVERITY_MAX)
        all_sev_mask.append(s_msk.cpu())

    probs  = torch.cat(all_probs).numpy().ravel()
    labels = torch.cat(all_labels).numpy().ravel().astype(int)
    sp     = torch.cat(all_sev_pred).numpy().ravel()
    st     = torch.cat(all_sev_true).numpy().ravel()
    sm     = torch.cat(all_sev_mask).numpy().ravel().astype(bool)

    auroc = roc_auc_score(labels, probs) if labels.sum() > 0 else 0.0
    mae   = float(np.abs(sp[sm] - st[sm]).mean()) if sm.sum() > 0 else -1.0

    return total_loss / len(loader), auroc, mae


# ── Checkpoint helpers ────────────────────────────────────────────────

def _save_best_ckpt(fold: int, ep: int, auroc: float, mae: float,
                    model, loss, best_auroc: float, ckpt_path: Path) -> float:
    """Overwrite best checkpoint if current AUROC improved. Returns new best."""
    if auroc > best_auroc:
        core = model.module if hasattr(model, "module") else model
        torch.save({
            "fold":       fold,
            "epoch":      ep,
            "auroc":      auroc,
            "mae":        mae,
            "model":      core.state_dict(),
            "loss":       loss.state_dict(),
        }, ckpt_path)
        print(f"    ✓ Best checkpoint saved  (AUROC {auroc:.4f}, MAE {mae:.1f})")
        return auroc
    return best_auroc


def _save_resume_ckpt(fold: int, stage: int, ep: int, best_auroc: float,
                      model, loss, opt, sch, scaler, cfg: "Config") -> None:
    """Rolling resume checkpoint – overwritten after every epoch."""
    core = model.module if hasattr(model, "module") else model
    torch.save({
        # ── position ────────────────────────────────────────────────
        "fold":        fold,
        "stage":       stage,
        "epoch":       ep,          # last *completed* epoch (0-indexed)
        "best_auroc":  best_auroc,
        # ── weights ─────────────────────────────────────────────────
        "model":       core.state_dict(),
        "loss":        loss.state_dict(),
        # ── optimiser / scheduler ───────────────────────────────────
        "opt":         opt.state_dict(),
        "sch":         sch.state_dict(),
        "scaler":      scaler.state_dict() if scaler is not None else None,
    }, cfg.CKPT_DIR / f"fold{fold}_resume.pt")


def _save_stage_final_ckpt(fold: int, stage: int, model, loss, cfg: "Config") -> None:
    """Snapshot at the boundary between stages – useful for debugging / rollback."""
    core = model.module if hasattr(model, "module") else model
    torch.save({
        "fold":  fold,
        "stage": stage,
        "model": core.state_dict(),
        "loss":  loss.state_dict(),
    }, cfg.CKPT_DIR / f"fold{fold}_stage{stage}_final.pt")
    print(f"    ↳ Stage {stage} final snapshot saved.")


def _load_resume(fold: int, cfg: "Config") -> dict | None:
    """Return resume dict if a resume checkpoint exists for this fold, else None."""
    p = cfg.CKPT_DIR / f"fold{fold}_resume.pt"
    if p.exists():
        ckpt = torch.load(p, map_location=cfg.DEVICE, weights_only=False)
        print(f"  ↺ Resume found: fold {fold+1}  stage {ckpt['stage']}  "
              f"epoch {ckpt['epoch']+1}  best_AUROC {ckpt['best_auroc']:.4f}")
        return ckpt
    return None


# ── Main fold runner ──────────────────────────────────────────────────

def run_fold(fold: int, df: pd.DataFrame,
             train_idx, val_idx, cfg: "Config") -> dict:
    print(f"\n{'='*60}")
    print(f"  FOLD {fold+1}/{cfg.N_FOLDS}")
    print(f"{'='*60}")

    train_loader, val_loader = make_loaders(df, train_idx, val_idx, cfg)
    print(f"  Train: {len(train_idx)}  |  Val: {len(val_idx)}")

    # ── Build model & loss ────────────────────────────────────────────
    _model = TBMTNet(cfg).to(cfg.DEVICE)
    _loss  = MultiTaskLoss(cfg.POS_WEIGHT, cfg.HUBER_BETA).to(cfg.DEVICE)

    scaler     = GradScaler("cuda") if (cfg.USE_AMP and cfg.AMP_DTYPE == "float16") else None
    ckpt_path  = cfg.CKPT_DIR / f"fold{fold}_best.pt"

    # ── Load resume state (if available) ─────────────────────────────
    resume     = _load_resume(fold, cfg)
    best_auroc = resume["best_auroc"] if resume else 0.0

    if resume is not None:
        _model.load_state_dict(resume["model"])
        _loss.load_state_dict(resume["loss"])

    # DataParallel must wrap AFTER loading state_dict
    if torch.cuda.device_count() > 1:
        _model = nn.DataParallel(_model)

    # Convenience: which stage/epoch did we last complete?
    resume_stage = resume["stage"] if resume else 0
    resume_epoch = resume["epoch"] if resume else -1   # -1 means "nothing done yet"

    # ── STAGE 1: classification only ─────────────────────────────────
    opt1 = make_optimizer(_model, cfg, stage=1, mtl_loss=_loss)
    sch1 = make_scheduler(opt1, cfg.S1_EPOCHS, cfg)

    if resume_stage == 1:
        # Restore exact optimiser + scheduler position mid-stage
        opt1.load_state_dict(resume["opt"])
        sch1.load_state_dict(resume["sch"])
        if resume["scaler"] and scaler:
            scaler.load_state_dict(resume["scaler"])

    if resume_stage <= 1:
        start_ep = (resume_epoch + 1) if resume_stage == 1 else 0
        print(f"\n  [Stage 1] Classification-only (reg head frozen)"
              f"{'  – resuming from ep ' + str(start_ep+1) if start_ep > 0 else ''}")
        for ep in range(start_ep, cfg.S1_EPOCHS):
            tl, cl, rl = train_epoch(_model, train_loader, opt1, sch1, _loss,
                                     scaler, cfg, ep, freeze_reg=True)
            vl, auroc, mae = validate(_model, val_loader, _loss, cfg)
            print(f"  S1 ep{ep+1:02d}  train={tl:.4f}  val={vl:.4f}  "
                  f"AUROC={auroc:.4f}  MAE={mae:.1f}")
            best_auroc = _save_best_ckpt(fold, ep, auroc, mae,
                                         _model, _loss, best_auroc, ckpt_path)
            _save_resume_ckpt(fold, 1, ep, best_auroc,
                              _model, _loss, opt1, sch1, scaler, cfg)

        _save_stage_final_ckpt(fold, 1, _model, _loss, cfg)

    # ── STAGE 2: full multi-task ──────────────────────────────────────
    opt2 = make_optimizer(_model, cfg, stage=2, mtl_loss=_loss)
    sch2 = make_scheduler(opt2, cfg.S2_EPOCHS, cfg)

    if resume_stage == 2:
        opt2.load_state_dict(resume["opt"])
        sch2.load_state_dict(resume["sch"])
        if resume["scaler"] and scaler:
            scaler.load_state_dict(resume["scaler"])

    if resume_stage <= 2:
        start_ep = (resume_epoch + 1) if resume_stage == 2 else 0
        print(f"\n  [Stage 2] Full multi-task (both heads)"
              f"{'  – resuming from ep ' + str(start_ep+1) if start_ep > 0 else ''}")
        for ep in range(start_ep, cfg.S2_EPOCHS):
            tl, cl, rl = train_epoch(_model, train_loader, opt2, sch2, _loss,
                                     scaler, cfg, ep, freeze_reg=False)
            vl, auroc, mae = validate(_model, val_loader, _loss, cfg)
            print(f"  S2 ep{ep+1:02d}  train={tl:.4f}  val={vl:.4f}  "
                  f"AUROC={auroc:.4f}  MAE={mae:.1f}  "
                  f"s_c={_loss.s_c.item():.3f} s_r={_loss.s_r.item():.3f}")
            best_auroc = _save_best_ckpt(fold, ep, auroc, mae,
                                         _model, _loss, best_auroc, ckpt_path)
            _save_resume_ckpt(fold, 2, ep, best_auroc,
                              _model, _loss, opt2, sch2, scaler, cfg)

        _save_stage_final_ckpt(fold, 2, _model, _loss, cfg)

    # ── STAGE 3: fine-tune + SWA ──────────────────────────────────────
    opt3      = make_optimizer(_model, cfg, stage=3, mtl_loss=_loss)
    sch3      = make_scheduler(opt3, cfg.S3_EPOCHS, cfg)
    swa_model = AveragedModel(_model)
    swa_sch   = SWALR(opt3, swa_lr=cfg.S3_LR)

    if resume_stage == 3:
        opt3.load_state_dict(resume["opt"])
        sch3.load_state_dict(resume["sch"])
        if resume["scaler"] and scaler:
            scaler.load_state_dict(resume["scaler"])

    start_ep = (resume_epoch + 1) if resume_stage == 3 else 0
    print(f"\n  [Stage 3] Fine-tune + SWA"
          f"{'  – resuming from ep ' + str(start_ep+1) if start_ep > 0 else ''}")
    for ep in range(start_ep, cfg.S3_EPOCHS):
        tl, cl, rl = train_epoch(_model, train_loader, opt3, sch3, _loss,
                                 scaler, cfg, ep, freeze_reg=False)
        if ep >= cfg.S3_EPOCHS - cfg.SWA_START:
            swa_model.update_parameters(_model)
            swa_sch.step()
        else:
            sch3.step(ep + 1)
        vl, auroc, mae = validate(_model, val_loader, _loss, cfg)
        print(f"  S3 ep{ep+1:02d}  train={tl:.4f}  val={vl:.4f}  "
              f"AUROC={auroc:.4f}  MAE={mae:.1f}")
        best_auroc = _save_best_ckpt(fold, ep, auroc, mae,
                                     _model, _loss, best_auroc, ckpt_path)
        _save_resume_ckpt(fold, 3, ep, best_auroc,
                          _model, _loss, opt3, sch3, scaler, cfg)

    _save_stage_final_ckpt(fold, 3, _model, _loss, cfg)

    # ── SWA: update batch-norm statistics ────────────────────────────
    class ImageOnlyLoader:
        """Thin wrapper so update_bn receives tensors, not dicts."""
        def __init__(self, loader): self.loader = loader
        def __iter__(self):
            for b in self.loader: yield b["image"]
        def __len__(self): return len(self.loader)

    torch.optim.swa_utils.update_bn(ImageOnlyLoader(train_loader), swa_model,
                                    device=cfg.DEVICE)

    # ── Fix: evaluate the SWA model and save if it beats current best ─
    # Previously the SWA model's updated weights + BN stats were computed
    # but then discarded — the fold best was always the last non-SWA epoch.
    # SWA averaging is most effective when we actually keep the averaged model.
    print("  Evaluating SWA model on validation set …")
    swa_core = swa_model.module if hasattr(swa_model, "module") else swa_model
    swa_core.eval()

    # Temporarily patch predict path: swa_model wraps the original module;
    # validate() calls model(imgs) which triggers AveragedModel.__call__ → OK.
    swa_val_loss, swa_auroc, swa_mae = validate(swa_model, val_loader, _loss, cfg)
    print(f"  SWA val:  loss={swa_val_loss:.4f}  AUROC={swa_auroc:.4f}  MAE={swa_mae:.1f}")

    if swa_auroc > best_auroc:
        # Extract the underlying averaged parameters (module inside AveragedModel)
        inner = swa_model.module if hasattr(swa_model, "module") else swa_model
        torch.save({
            "fold":   fold,
            "epoch":  "swa",
            "auroc":  swa_auroc,
            "mae":    swa_mae,
            "model":  inner.state_dict(),
            "loss":   _loss.state_dict(),
        }, ckpt_path)
        best_auroc = swa_auroc
        print(f"    ✓ SWA checkpoint saved as fold best  (AUROC {swa_auroc:.4f})")
    else:
        print(f"    ↳ SWA AUROC ({swa_auroc:.4f}) did not beat current best "
              f"({best_auroc:.4f}) — keeping Stage-3 checkpoint.")

    # ── Fold complete – remove rolling resume file ────────────────────
    resume_p = cfg.CKPT_DIR / f"fold{fold}_resume.pt"
    if resume_p.exists():
        resume_p.unlink()
        print(f"  ✓ Resume checkpoint cleaned up.")

    print(f"\n  Fold {fold+1} best val AUROC = {best_auroc:.4f}")
    return {"fold": fold, "best_auroc": best_auroc, "ckpt": str(ckpt_path)}


# ── Cross-validation entry point ─────────────────────────────────────
def run_training(df: pd.DataFrame, cfg: "Config") -> List[dict]:
    cv = StratifiedGroupKFold(n_splits=cfg.N_FOLDS, shuffle=True,
                              random_state=cfg.SEED)
    X      = np.arange(len(df))
    y      = df["strat_key"].values
    groups = df["patient_id"].values

    results = []
    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y, groups)):
        res = run_fold(fold, df, tr_idx, va_idx, cfg)
        results.append(res)
        print(f"\nFold {fold+1} summary: AUROC={res['best_auroc']:.4f}")

    mean_auroc = np.mean([r["best_auroc"] for r in results])
    print(f"\n{'='*60}")
    print(f"  5-Fold CV mean AUROC = {mean_auroc:.4f}")
    print(f"{'='*60}")
    return results


fold_results = run_training(labels_df, CFG)


## Cell 8 — OOF Evaluation + Plots

In [ ]:
"""Cell 8 – OOF Evaluation: AUROC, sensitivity/specificity/F1, MAE/RMSE/Pearson."""

def youden_threshold(fpr, tpr, thresholds):
    """Optimal threshold by Youden's J = TPR - FPR."""
    j = tpr - fpr
    return thresholds[np.argmax(j)]


def bootstrap_auroc(y_true, y_score, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        if y_true[idx].sum() == 0 or y_true[idx].sum() == len(idx):
            continue
        aucs.append(roc_auc_score(y_true[idx], y_score[idx]))
    aucs = np.array(aucs)
    return np.percentile(aucs, 2.5), np.percentile(aucs, 97.5)


@torch.no_grad()
def predict_fold(fold: int, df: pd.DataFrame, val_idx, cfg: "Config"):
    """Load best checkpoint for a fold and run inference on val set."""
    ckpt_path = cfg.CKPT_DIR / f"fold{fold}_best.pt"
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

    _model = TBMTNet(cfg).to(cfg.DEVICE)
    ckpt   = torch.load(ckpt_path, map_location=cfg.DEVICE, weights_only=False)
    state_dict = ckpt["model"]
    state_dict = {k.replace("module.", "") if k.startswith("module.") else k: v for k, v in state_dict.items()}
    _model.load_state_dict(state_dict)
    _model.eval()

    ds     = TBXDataset(df.iloc[val_idx], cfg, train=False)
    loader = DataLoader(ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                        num_workers=cfg.NUM_WORKERS, pin_memory=True)

    probs_list, labels_list, sev_pred_list, sev_true_list, sev_mask_list = \
        [], [], [], [], []
    core = _model.module if hasattr(_model, "module") else _model

    for batch in loader:
        imgs  = batch["image"].to(cfg.DEVICE)
        logit, sev = core(imgs)
        probs_list.append(torch.sigmoid(logit).cpu())
        labels_list.append(batch["tb_label"])
        sev_pred_list.append(sev.cpu() * cfg.SEVERITY_MAX)
        sev_true_list.append(batch["severity"] * cfg.SEVERITY_MAX)
        sev_mask_list.append(batch["severity_mask"])

    return {
        "probs":    torch.cat(probs_list).numpy().ravel(),
        "labels":   torch.cat(labels_list).numpy().ravel().astype(int),
        "sev_pred": torch.cat(sev_pred_list).numpy().ravel(),
        "sev_true": torch.cat(sev_true_list).numpy().ravel(),
        "sev_mask": torch.cat(sev_mask_list).numpy().ravel().astype(bool),
    }


def evaluate_oof(df: pd.DataFrame, fold_results: List[dict], cfg: "Config"):
    """Aggregate OOF predictions across all folds and compute metrics."""
    cv = StratifiedGroupKFold(n_splits=cfg.N_FOLDS, shuffle=True,
                              random_state=cfg.SEED)
    X      = np.arange(len(df))
    y      = df["strat_key"].values
    groups = df["patient_id"].values

    all_probs  = np.zeros(len(df))
    all_labels = np.zeros(len(df), dtype=int)
    all_sp     = np.zeros(len(df))
    all_st     = np.zeros(len(df))
    all_sm     = np.zeros(len(df), dtype=bool)

    for fold, (_, val_idx) in enumerate(cv.split(X, y, groups)):
        preds = predict_fold(fold, df, val_idx, cfg)
        all_probs[val_idx]  = preds["probs"]
        all_labels[val_idx] = preds["labels"]
        all_sp[val_idx]     = preds["sev_pred"]
        all_st[val_idx]     = preds["sev_true"]
        all_sm[val_idx]     = preds["sev_mask"]

    # ── Classification metrics ────────────────────────────────────────
    auroc         = roc_auc_score(all_labels, all_probs)
    ci_lo, ci_hi  = bootstrap_auroc(all_labels, all_probs)
    fpr, tpr, thr = roc_curve(all_labels, all_probs)
    best_thr      = youden_threshold(fpr, tpr, thr)
    preds_bin     = (all_probs >= best_thr).astype(int)
    cm            = confusion_matrix(all_labels, preds_bin)
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn + 1e-9)
    spec = tn / (tn + fp + 1e-9)
    f1   = f1_score(all_labels, preds_bin)

    # Partial AUROC (specificity >= 0.70)
    fpr_thresh = 1 - 0.70
    mask_spec  = fpr <= fpr_thresh
    p_auroc    = np.trapz(tpr[mask_spec], fpr[mask_spec]) / fpr_thresh \
                 if mask_spec.sum() > 1 else 0.0

    print("\n" + "="*55)
    print("  OOF Classification Results")
    print("="*55)
    print(f"  AUROC          : {auroc:.4f}  (95% CI [{ci_lo:.4f}, {ci_hi:.4f}])")
    print(f"  Partial AUROC  : {p_auroc:.4f}  (spec >= 0.70 region)")
    print(f"  Threshold (J)  : {best_thr:.4f}")
    print(f"  Sensitivity    : {sens:.4f}")
    print(f"  Specificity    : {spec:.4f}")
    print(f"  F1             : {f1:.4f}")
    print(f"  Confusion Mat  :\n    TN={tn} FP={fp}\n    FN={fn} TP={tp}")

    # ── Severity metrics ──────────────────────────────────────────────
    sv_pred = all_sp[all_sm]; sv_true = all_st[all_sm]
    if len(sv_pred) > 0:
        mae  = float(np.abs(sv_pred - sv_true).mean())
        rmse = float(np.sqrt(((sv_pred - sv_true)**2).mean()))
        pr, _ = pearsonr(sv_pred, sv_true)
        sp_r, _ = spearmanr(sv_pred, sv_true)
        print("\n  OOF Severity Results")
        print("="*55)
        print(f"  MAE            : {mae:.2f}  (target < 15)")
        print(f"  RMSE           : {rmse:.2f}")
        print(f"  Pearson r      : {pr:.4f}  (target > 0.75)")
        print(f"  Spearman rho   : {sp_r:.4f}")
        print(f"  N severity cases: {len(sv_pred)}")
    else:
        print("\n  [WARN] No valid severity cases for regression metrics.")

    # ── Plots ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ROC curve
    ax = axes[0]
    ax.plot(fpr, tpr, lw=2, color="#4F46E5",
            label=f"AUROC = {auroc:.4f} [{ci_lo:.3f}–{ci_hi:.3f}]")
    ax.plot([0,1],[0,1],"k--", lw=1)
    ax.axvline(1-0.70, color="grey", ls=":", label="spec=0.70 (WHO TPP)")
    ax.scatter([1-spec],[sens], color="red", zorder=5, label=f"Youden J ({best_thr:.3f})")
    ax.set(xlabel="FPR", ylabel="TPR", title="ROC Curve (OOF)")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    # Reliability diagram (severity)
    ax = axes[1]
    if len(sv_pred) > 5:
        bins = np.linspace(0, 140, 11)
        bin_centres, bin_means, bin_counts = [], [], []
        for lo, hi in zip(bins[:-1], bins[1:]):
            mask = (sv_true >= lo) & (sv_true < hi)
            if mask.sum() > 0:
                bin_centres.append((lo + hi) / 2)
                bin_means.append(sv_pred[mask].mean())
                bin_counts.append(mask.sum())
        ax.scatter(bin_centres, bin_means, s=[c*3 for c in bin_counts],
                   color="#10B981", alpha=0.8, label="Pred mean per bin")
        ax.plot([0,140],[0,140],"k--", lw=1, label="Perfect calibration")
        ax.set(xlabel="True Timika", ylabel="Predicted Timika",
               title="Severity Reliability Diagram", xlim=(0,140), ylim=(0,140))
        ax.legend(fontsize=9); ax.grid(alpha=0.3)
        ax.text(5, 130, f"MAE={mae:.1f}  r={pr:.3f}", fontsize=10)
    else:
        ax.text(0.5, 0.5, "Insufficient severity data", ha="center",
                transform=ax.transAxes)

    plt.tight_layout()
    plot_path = CFG.BASE / "oof_evaluation.png"
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print(f"\nPlot saved → {plot_path}")


evaluate_oof(labels_df, fold_results, CFG)


## Cell 9 — TTA + Ensemble Inference

In [ ]:
"""Cell 9 – TTA + 5-Fold Ensemble Inference → submission.csv"""

import torchvision.transforms.functional as TF   # for rotation TTA

# ── Four deterministic TTA augmentations ─────────────────────────────
# All keep spatial size at IMAGE_SIZE×IMAGE_SIZE so the fixed positional
# embedding (196 patch tokens) is never invalidated.
#   1) original
#   2) horizontal flip   – most useful for CXR (L/R mirror is clinically valid)
#   3) +5° rotation      – mild geometric jitter
#   4) −5° rotation
_TTA_FNS = [
    lambda x: x,
    lambda x: torch.flip(x, dims=[-1]),
    lambda x: TF.rotate(x, angle=5,  fill=0),
    lambda x: TF.rotate(x, angle=-5, fill=0),
]


@torch.no_grad()
def predict_tta(model: nn.Module, imgs: torch.Tensor,
                cfg: "Config", n_tta: int = 4) -> Tuple[torch.Tensor, torch.Tensor]:
    """Multi-augmentation TTA: original, h-flip, ±5° rotation (4 passes by default).

    All augmentations keep spatial dimensions at IMAGE_SIZE so the transformer's
    fixed pos-embed (196 patch tokens) stays valid throughout.
    Set n_tta=2 to fall back to original + h-flip only.
    """
    core = model.module if hasattr(model, "module") else model
    prob_sum = torch.zeros(imgs.size(0), 1, device=cfg.DEVICE)
    sev_sum  = torch.zeros(imgs.size(0), 1, device=cfg.DEVICE)

    augs = _TTA_FNS[:n_tta]
    for aug_fn in augs:
        x = aug_fn(imgs.clone())
        logit, sev = core(x)
        prob_sum += torch.sigmoid(logit)
        sev_sum  += sev * cfg.SEVERITY_MAX

    return prob_sum / len(augs), sev_sum / len(augs)


def load_fold_model(fold: int, cfg: "Config") -> nn.Module:
    ckpt_path = cfg.CKPT_DIR / f"fold{fold}_best.pt"
    _model = TBMTNet(cfg).to(cfg.DEVICE)
    ckpt   = torch.load(ckpt_path, map_location=cfg.DEVICE, weights_only=False)
    state_dict = ckpt["model"]
    state_dict = {k.replace("module.", "") if k.startswith("module.") else k: v for k, v in state_dict.items()}
    _model.load_state_dict(state_dict)
    _model.eval()
    return _model


def ensemble_predict(df: pd.DataFrame, cfg: "Config",
                     n_tta: int = 2) -> pd.DataFrame:
    """
    Run all N_FOLDS models with TTA; average predictions.
    Returns df with columns: image_path, tb_prob, timika_score
    """
    ds     = TBXDataset(df, cfg, train=False)
    loader = DataLoader(ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                        num_workers=cfg.NUM_WORKERS, pin_memory=True)

    fold_probs = []
    fold_sevs  = []

    for fold in range(cfg.N_FOLDS):
        ckpt = cfg.CKPT_DIR / f"fold{fold}_best.pt"
        if not ckpt.exists():
            print(f"  [WARN] Fold {fold} checkpoint not found — skipping")
            continue

        print(f"  Running fold {fold+1}/{cfg.N_FOLDS} …", end=" ", flush=True)
        _model = load_fold_model(fold, cfg)

        probs_list, sevs_list = [], []
        for batch in loader:
            imgs = batch["image"].to(cfg.DEVICE)
            if cfg.USE_AMP:
                with autocast("cuda", dtype=getattr(torch, cfg.AMP_DTYPE)):
                    prob, sev = predict_tta(_model, imgs, cfg, n_tta)
            else:
                prob, sev = predict_tta(_model, imgs, cfg, n_tta)
            probs_list.append(prob.cpu())
            sevs_list.append(sev.cpu())

        fold_probs.append(torch.cat(probs_list).numpy().ravel())
        fold_sevs.append(torch.cat(sevs_list).numpy().ravel())
        print("done")

    if not fold_probs:
        raise RuntimeError("No fold checkpoints found.")

    # Mean ensemble
    ens_probs_raw = np.stack(fold_probs, axis=0).mean(axis=0)
    ens_sevs_raw  = np.stack(fold_sevs,  axis=0).mean(axis=0)

    # 1. Recalibrate probabilities to undo POS_WEIGHT inflation
    # Math: p = q / (w * (1 - q) + q)
    w = cfg.POS_WEIGHT
    ens_probs_calibrated = ens_probs_raw / (w * (1 - ens_probs_raw) + ens_probs_raw)

    result_df = df[["image_path", "patient_id", "tb_label"]].copy()
    result_df["tb_prob"]      = ens_probs_calibrated
    result_df["tb_pred"]      = (ens_probs_calibrated >= 0.5).astype(int)
    
    # 2. Fix Timika Score (Model wasn't trained on normal lungs, so force to 0 if Normal)
    result_df["timika_score"] = ens_sevs_raw.clip(0, 140) * result_df["tb_pred"]
    return result_df


def save_submission(result_df: pd.DataFrame, cfg: "Config") -> None:
    sub = result_df[["image_path", "tb_prob", "timika_score"]].copy()
    sub["image_id"] = sub["image_path"].apply(lambda p: Path(p).stem)
    sub = sub[["image_id", "tb_prob", "timika_score"]]
    sub.to_csv(cfg.BASE / "submission.csv", index=False)
    print(f"submission.csv saved → {cfg.BASE / 'submission.csv'}")
    print(sub.head(10))


def print_summary(result_df: pd.DataFrame) -> None:
    pos = (result_df["tb_pred"] == 1).sum()
    neg = (result_df["tb_pred"] == 0).sum()
    print("\n" + "="*50)
    print("  ENSEMBLE INFERENCE SUMMARY")
    print("="*50)
    print(f"  Total images    : {len(result_df)}")
    print(f"  Predicted TB+   : {pos}  ({100*pos/len(result_df):.1f}%)")
    print(f"  Predicted TB-   : {neg}  ({100*neg/len(result_df):.1f}%)")
    if result_df["timika_score"].notna().any():
        s = result_df.loc[result_df["tb_pred"]==1, "timika_score"]
        print(f"  Timika (TB+)    : mean={s.mean():.1f}  "
              f"median={s.median():.1f}  "
              f"range=[{s.min():.1f}, {s.max():.1f}]")

    # GradCAM visualisation on top-5 uncertain cases
    result_df["uncertainty"] = (result_df["tb_prob"] - 0.5).abs()
    uncertain_idx = result_df["uncertainty"].nsmallest(5).index.tolist()
    print(f"\n  Top-5 most uncertain cases (prob closest to 0.5):")
    print(result_df.loc[uncertain_idx,
                        ["image_id" if "image_id" in result_df.columns
                         else "image_path", "tb_prob", "timika_score"]].to_string())


# ── Run inference on full dataset ─────────────────────────────────────
print("Running 5-fold TTA ensemble inference …")
result_df = ensemble_predict(labels_df, CFG, n_tta=4)   # 4 TTA passes: orig, h-flip, ±5° rot
print_summary(result_df)
save_submission(result_df, CFG)
